In [1]:
import os
from io import StringIO 

print('Machine Learning Assignment2')
print('bitid: 2024dc04016')
print('Name: Jeya Prakash S')
#Read the Kaggle and AWS keys from keys.txt file, then load to env varaibles
keys_file = open('keys.txt', 'r')
Lines = keys_file.readlines()

keys = []
for line in Lines:
  line = line.strip()
  keys.append(line)
  
KAGGLE_USERNAME = keys[0]
KAGGLE_KEY = keys[1]

#Load keys to env varaibles
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

print(os.environ['KAGGLE_USERNAME'])
print(os.environ['KAGGLE_KEY'])


jp
KGAT_ce9528a4398a381e444edbd2d35824b5


In [2]:
 !pip install kaggle
#Read the dataset from Kaggle
from kaggle.api.kaggle_api_extended import KaggleApi


#https://www.kaggle.com/datasets/jpdragons/pulmanory-heart-dataset

dataset = 'jpdragons/pulmanory-heart-dataset'
path = 'datasets/pulmanory-heart-dataset'
dataset_url = f"https://www.kaggle.com/datasets/{dataset}"

api = KaggleApi()
api.authenticate()
api.dataset_download_files(dataset, 'data', path, unzip=True)

print ('file downloaded successfully from kaggle ')




Dataset URL: https://www.kaggle.com/datasets/jpdragons/pulmanory-heart-dataset
file downloaded successfully from kaggle 


In [3]:
#Read the data into Python dataframe
   
import pandas as pd
pulmanodata = pd.read_csv('data/Pulmanory heart Disease.csv')
print(pulmanodata.head(2))
print('data csv read')
print('total rows :',len(pulmanodata))
   


   Unnamed: 0  age  sex  chest pain type  resting bps  cholesterol  \
0           0   40    1                2          140        289.0   
1           1   49    0                3          160        180.0   

   fasting blood sugar  resting ecg  max heart rate  exercise angina  oldpeak  \
0                    0            0             172                0      0.0   
1                    0            0             156                0      1.0   

   ST slope  target  
0         1       0  
1         2       1  
data csv read
total rows : 1048


In [4]:
 !pip install xgboost

In [5]:
#all model imports important starting step
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

In [6]:
#Separate Features & Target
X = pulmanodata.drop("target", axis=1)
y = pulmanodata["target"]

In [7]:
#Train-Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:

#Feature Scaling (ONLY ON TRAIN DATA)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
# Evaluation Function (ALL METRICS)
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

In [10]:
#1Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)

lr_metrics = evaluate_model(lr, X_test_scaled, y_test)


In [11]:
#2DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

dt_metrics = evaluate_model(dt, X_test, y_test)



In [12]:
#3KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

knn_metrics = evaluate_model(knn, X_test_scaled, y_test)


In [13]:
#4 Naive Bayes (Gaussian)

nb = GaussianNB()
nb.fit(X_train_scaled, y_train)

nb_metrics = evaluate_model(nb, X_test_scaled, y_test)


In [14]:
#5 Random Forest (Ensemble)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf.fit(X_train, y_train)

rf_metrics = evaluate_model(rf, X_test, y_test)



In [15]:
#6 XGBoost (Ensemble)

xgb = XGBClassifier(
    #use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
xgb.fit(X_train, y_train)

xgb_metrics = evaluate_model(xgb, X_test, y_test)


In [16]:
#outputs (Combine into DataFrame)
results = pd.DataFrame.from_dict({
    "Logistic Regression": lr_metrics,
    "Decision Tree": dt_metrics,
    "KNN": knn_metrics,
    "Naive Bayes": nb_metrics,
    "Random Forest": rf_metrics,
    "XGBoost": xgb_metrics
}, orient="index")

print("\nMODEL COMPARISON TABLE")
print(results)


MODEL COMPARISON TABLE
                     Accuracy       AUC  Precision    Recall        F1  \
Logistic Regression  0.790476  0.856948   0.788462  0.788462  0.788462   
Decision Tree        0.766667  0.766600   0.766990  0.759615  0.763285   
KNN                  0.838095  0.883119   0.824074  0.855769  0.839623   
Naive Bayes          0.733333  0.795083   0.750000  0.692308  0.720000   
Random Forest        0.838095  0.905615   0.830189  0.846154  0.838095   
XGBoost              0.814286  0.884253   0.792793  0.846154  0.818605   

                          MCC  
Logistic Regression  0.580914  
Decision Tree        0.533273  
KNN                  0.676770  
Naive Bayes          0.467592  
Random Forest        0.676343  
XGBoost              0.630174  


In [18]:
#Save Models (CRITICAL FOR STREAMLIT)
import joblib
import os

os.makedirs("model", exist_ok=True)
joblib.dump(lr, "model/logistic.pkl")
joblib.dump(dt, "model/decision_tree.pkl")
joblib.dump(knn, "model/knn.pkl")
joblib.dump(nb, "model/naive_bayes.pkl")
joblib.dump(rf, "model/random_forest.pkl")
joblib.dump(xgb, "model/xgboost.pkl")
#joblib.dump(scaler, "model/scaler.pkl")

print("All models saved successfully.")


All models saved successfully.


In [35]:
import pandas as pd

# -----------------------------

# -----------------------------
#  Observations
observations = {
    "ML Model Name             ":"|Observation about model performance",
    "-------------------------------------------------------------------------------------------------------------------------------------":"",
    "Logistic Regression       ":"|Performs well on linearly separable features. Fast training but may underperform on non-linear relationships.",
    "Decision Tree             ":"|Easy to interpret, can overfit on training data if not tuned. Moderate accuracy on test set.",
    "KNN                       ":"|Sensitive to feature scaling. Performance depends on choice of K and distance metric.",
    "Naive Bayes               ":"|Assumes feature independence, computationally efficient but may be less accurate if features are correlated.",
    "Random Forest (Ensemble)  ":"|Handles non-linearity and reduces overfitting. Performs very well on most metrics.",
    "XGBoost (Ensemble)        ":"|Best overall performance due to boosting. Strong generalization and high AUC."
}

# -----------------------------
#  Generate README content
readme_content = f"""# Pulmanory Heart Disease Prediction using ML

---

## a. Problem Statement

The objective of this project is to build and compare multiple machine learning classification models to predict the presence or absence of heart disease based on clinical features. The project also demonstrates deployment of these models in an interactive Streamlit web application.

---

## b. Dataset Description

- **Dataset Name:** Heart Disease Dataset (UCI / Kaggle)
- **Number of Instances:** 900+
- **Number of Features:** 13
- **Target Variable:** `target` (0 = No heart disease, 1 = Heart disease)
- **Type:** Binary Classification
- **Description:** The dataset contains clinical attributes such as age, sex, blood pressure, cholesterol, maximum heart rate, and others, which are used to predict the presence of heart disease.

---

## c. Models Used and Evaluation Metrics

{results.to_markdown()}

---

##  Observations on Model Performance
"""

for model, obs in observations.items():
    readme_content += f"\n- **{model}**: {obs}"

readme_content += "\n\n---\n\n## How to Run\n\n1. Clone the repository.\n2. Install dependencies:\n\n```bash\npip install -r requirements.txt\n```\n\n3. Run the Streamlit app:\n\n```bash\nstreamlit run app.py\n```\n\n4. Upload a test CSV file and select a model from the dropdown to see predictions and evaluation metrics.\n"

# -----------------------------
#  Write README.md
with open("README.md", "w") as f:
    f.write(readme_content)

print(" README.md generated successfully!")


 README.md generated successfully!
